# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [4]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Intended chart:** Line chart, indexed to 1970, Africa highlighted

**Why:** Change over time on a numeric → line chart. Indexing to 1970 makes growth comparable across continents of very different sizes; grey + one accent answers "fastest".

In [1]:
growth = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
growth['Index'] = growth['Population'] / growth.groupby('Continent')['Population'].transform('first')   # 1970 = 1

colours = {c: 'darkorange' if c == 'Africa' else 'grey' for c in growth['Continent'].unique()}

# Put Africa last so it is drawn on top of the grey lines
order = [c for c in growth['Continent'].unique() if c != 'Africa'] + ['Africa']

fig = px.line(growth, x='Year', y='Index',
              color='Continent',
              color_discrete_map=colours,
              category_orders={'Continent': order},
              markers=False,
              title="Africa's population has nearly quadrupled since 1970, , far outpacing every other continent",
              template='plotly_white',
              height=700, width=1000)

# Thin grey context lines, thick Africa line
fig.update_traces(line_width=1.5)
fig.update_traces(line_width=4, selector=dict(name='Africa'))

# Label Africa
end = growth.loc[growth['Year'] == 2022]
africa_y = end.loc[end['Continent'] == 'Africa', 'Index'].iloc[0]
fig.add_annotation(x=2022, y=africa_y,
                   text=f"Africa ×{africa_y:.1f}",
                   xanchor='left', xshift=8, showarrow=False,
                   font=dict(color='darkorange', size=14))

# Label the grey group as a whole, next to the highest grey line
others_y = end.loc[end['Continent'] != 'Africa', 'Index'].max()
fig.add_annotation(x=2022, y=others_y,
                   text="Other continents",
                   xanchor='left', xshift=8, showarrow=False,
                   font=dict(color='grey', size=13))

fig.update_xaxes(showgrid=False, title='')
fig.update_yaxes(showgrid=False, ticksuffix='×',
                 title='Population relative to 1970 (1970 = 1×)')
fig.update_layout(showlegend=False, margin=dict(r=140))
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Intended chart:** Sorted horizontal bar chart, African countries highlighted

**Why:** Ranking 15 countries on one measure → sorted horizontal bar w/ zero baseline, readable labels. Highlighting the African countries answers the second half of the question.

In [ ]:
df['Added'] = df['2022 Population'] - df['1970 Population']
top15 = df.nlargest(15, 'Added')

top15['Group'] = np.where(top15['Continent'] == 'Africa', 'Africa', 'Rest of the world')

fig = px.bar(top15, x='Added', y='Country', color='Group', orientation='h', text_auto='.3s',
             category_orders={'Country': top15['Country'].tolist()},     # keep the ranking across both colours
             color_discrete_map={'Africa': 'darkorange', 'Rest of the world': 'lightgrey'},
             title='India and China added 1.5 billion people since 1970 — 4 of the top 15 growers are in Africa',
             labels={'Added': 'People added 1970–2022', 'Country': '', 'Group': ''},
             template='plotly_white', height=550, width=950)
fig.update_traces(textposition='outside')
fig.update_xaxes(visible=False)
fig.update_layout(title_x=0.01, bargap=0.3, legend=dict(orientation='h', x=0, y=1.1))
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Intended chart:** Scatter plot, log–log axes, coloured by continent, outliers annotated

**Why:** Relationship between two numerical variables → scatter plot. Both variables span several orders of magnitude, so log axes are needed.

In [ ]:
fig = px.scatter(df, x='Area', y='2022 Population', color='Continent', hover_name='Country',
                 log_x=True, log_y=True,
                 color_discrete_sequence=px.colors.qualitative.Safe,          # CVD-safe qualitative palette
                 title='Bigger countries hold more people — but density varies enormously',
                 labels={'Area': 'Land area (km², log scale)', '2022 Population': 'Population 2022 (log scale)'},
                 template='plotly_white', height=600, width=1000)

# Arrow offsets in pixels (ax = horizontal, ay = vertical; positive ay = label below the point)
offsets = {
    'Singapore':      (-40, -40),
    'Monaco':         ( 45, -30),
    'Mongolia':       ( 45,  40),
    'Western Sahara': (-60,  40),
    'Greenland':      (0,  50),
    'Russia':         (10, -35),
}

for country, (ax, ay) in offsets.items():
    row = df.loc[df['Country'] == country].iloc[0]
    fig.add_annotation(x=np.log10(row['Area']), y=np.log10(row['2022 Population']),   # log axes need log10 coords
                       text=country,
                       showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1, arrowcolor='grey',
                       ax=ax, ay=ay,
                       font=dict(size=12))

fig.update_traces(marker=dict(size=9, opacity=0.8))
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.update_layout(title_x=0.01,
                  legend=dict(title=None, orientation='h',
                              yanchor='bottom', y=1.02,       # just above the plot, under the title
                              xanchor='left', x=0),
                  margin=dict(t=110))
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Intended chart:** Box/violin plot per continent, sorted by median, log axis due to the huge spread, in this case use box plots

**Why:** Comparing distributions across 6 groups → box/violin plot: medians, spread and outliers side by side. Density is extremely skewed, so a log axis is needed; in this case box plots is the only option

In [ ]:
order = df.groupby('Continent')['Density'].median().sort_values().index.tolist()

fig = px.box(df, x='Density', y='Continent', points='outliers', orientation='h', 
             hover_name='Country', 
             log_x=True,
             category_orders={'Continent': order}, color_discrete_sequence=['steelblue'],
             title='South America is the least crowded continent — North America is the densest',
             labels={'Density': 'People per km² (log scale)', 'Continent': ''},
             template='plotly_white', height=500, width=1000)
fig.update_xaxes(showgrid=False)
fig.update_layout(title_x=0.01)
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Intended chart:** Choropleth map with a diverging scale centred on 0%

**Why:** "Where in the world" → choropleth. Growth and decline are two directions from zero, so a diverging, CVD-safe scale centred on 0 is the right encoding.

In [ ]:
df['Change'] = (df['2022 Population'] / df['2000 Population'] - 1) * 100

fig = px.choropleth(df, locations='CCA3', locationmode='ISO-3', color='Change', hover_name='Country',
                    color_continuous_scale='RdBu', color_continuous_midpoint=0, range_color=[-60, 60],
                    title='Africa and the Gulf grew fastest since 2000 — Eastern Europe is shrinking',
                    labels={'Change': 'Change 2000–2022 (%)'}, template='plotly_white', height=550, width=1100)

fig.update_geos(showframe=False, showcoastlines=False)

fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Intended chart:** Treemap (World → Continent → Country), coloured by continent

**Why:** Part-to-whole with a two-level hierarchy and 234 countries → treemap

In [ ]:
fig = px.treemap(df, path=[px.Constant('World'), 'Continent', 'Country'],
                 values='2022 Population', color='Continent',
                 color_discrete_sequence=px.colors.qualitative.Safe,
                 color_discrete_map={'(?)': 'lightgrey'},       # grey the World root
                 title='Asia holds 59% of humanity — China and India alone are 36% of the world\'s population',
                 template='plotly_white', height=650, width=1100)

fig.update_traces(texttemplate='%{label}<br>%{percentRoot:.1%}', root_color='lightgrey')

fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Intended chart:** 100% stacked bar chart (Africa on the baseline, accent colour)

**Why:** Composition (4 parts) compared across 8 years → 100% stacked bar. Grouping the small continents keeps it to 4 segments; putting Africa first places it on the common baseline

In [5]:
df_long['Region'] = np.where(df_long['Continent'].isin(['Africa', 'Asia', 'Europe']), df_long['Continent'], 'Rest of the world')
shares = df_long.groupby(['Year', 'Region'], as_index=False)['Population'].sum()
shares['Share'] = shares['Population'] / shares.groupby('Year')['Population'].transform('sum')
shares['Year'] = shares['Year'].astype(str)                                  # one bar per census year

fig = px.bar(shares, x='Year', y='Share', color='Region', text_auto='.0%',
             category_orders={'Region': ['Africa', 'Europe', 'Asia', 'Rest of the world']},   # Africa on the baseline
             color_discrete_map={'Africa': 'darkorange', 'Europe': 'steelblue', 'Asia': 'slategrey', 'Rest of the world': 'lightgrey'},
             title="Africa's share of the world population has nearly doubled since 1970,; Europe's shrank by almost as much",
             labels={'Year': '', 'Region': ''}, template='plotly_white', height=550, width=950)
fig.update_yaxes(visible=False)
fig.update_layout(bargap=0.25, legend=dict(orientation='h', x=0, y=1.06, traceorder='normal'))
fig.show()